# CMB-0030 — Galaxy Inspector
Preserves the working CMB-0029 Aladin viewer and adds catalog identification, Planck18 cosmology, nearby objects, archive/publication searches, notes, and MY_GALAXIES.csv export.

In [ ]:
# CMB-0030 setup
!pip -q install astropy astroquery pandas ipywidgets requests


In [ ]:
# CMB-0030
from __future__ import annotations
import html,re,urllib.parse
from datetime import datetime
from pathlib import Path
import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display,FileLink,clear_output
from astropy import units as u
from astropy.coordinates import SkyCoord
from astropy.cosmology import Planck18
from astroquery.ipac.ned import Ned
from astroquery.simbad import Simbad

VERSION='CMB-0030'
DEFAULT_TARGET='03 32 39.99 -27 48 00.0'
DEFAULT_FOV=0.08
CATALOG_PATH=Path('/content/MY_GALAXIES.csv')
SURVEYS=[('DSS2 color','P/DSS2/color'),('DSS2 blue','P/DSS2/blue'),('2MASS color infrared','P/2MASS/color'),('WISE color infrared','P/allWISE/color'),('GALEX GR6/7 ultraviolet','P/GALEXGR6/AIS/color'),('Hubble Outreach color','CDS/P/HST/EPO'),('Hubble GOODS color','CDS/P/HST/GOODS/color'),('Hubble GOODS i-band','CDS/P/HST/GOODS/i'),('Hubble R-band','CDS/P/HST/R'),('Hubble H-beta','CDS/P/HST/Hbeta'),('JWST F150W','CDS/P/JWST/F150W'),('JWST F444W','CDS/P/JWST/F444W'),('JWST F480M','CDS/P/JWST/F480M')]
STATE={'coord':None,'object':None,'nearby':pd.DataFrame()}

def aladin_url(t,f,s):
    return 'https://aladin.u-strasbg.fr/AladinLite/?'+urllib.parse.urlencode({'target':t.strip(),'fov':f'{float(f):g}','survey':s})

def selected_name():
    return next((a for a,b in SURVEYS if b==survey.value),survey.value)

def viewer_html(t,f,s,n):
    url=aladin_url(t,f,s)
    return f'''<div style="border:1px solid #78909c;border-radius:10px;overflow:hidden;background:#000"><div style="padding:8px 12px;background:#0b172a;color:white;font-family:sans-serif"><b>{html.escape(n)}</b><br><span style="font-size:12px;opacity:.85">Target: {html.escape(t)} · FOV {float(f):g}°</span></div><iframe src="{html.escape(url)}" style="width:100%;height:820px;border:0;display:block;background:#000;touch-action:auto" allowfullscreen></iframe></div>'''

def parse_target(v):
    v=v.strip(); p=re.split(r'[ ,]+',v)
    if len(p)==2:
        try:return SkyCoord(float(p[0])*u.deg,float(p[1])*u.deg,frame='icrs')
        except:pass
    try:return SkyCoord(v,unit=(u.hourangle,u.deg),frame='icrs')
    except:return SkyCoord.from_name(v)

def sf(v):
    try:
        if v is None or np.ma.is_masked(v):return np.nan
        return float(v)
    except:return np.nan

def first(row,names,default=np.nan):
    cols=set(getattr(row,'colnames',[]) or getattr(row,'index',[]))
    for n in names:
        if n in cols and not np.ma.is_masked(row[n]):return row[n]
    return default

def nearest_ned(c,radius=2):
    t=Ned.query_region(c,radius=radius*u.arcmin)
    rows=[]
    if t is not None:
        for r in t:
            ra=sf(first(r,['RA','RA(deg)'])); dec=sf(first(r,['DEC','DEC(deg)']))
            if not np.isfinite(ra+dec):continue
            q=SkyCoord(ra*u.deg,dec*u.deg)
            rows.append({'Object ID':str(first(r,['Object Name','Object_Name','MAIN_ID'],'Unknown')),'Catalog':'NASA/IPAC Extragalactic Database (NED)','RA':ra,'Dec':dec,'Angular separation (arcsec)':c.separation(q).arcsec,'Redshift':sf(first(r,['Redshift','z'])),'Type':str(first(r,['Type','Object Type'],''))})
    if not rows:return None,pd.DataFrame()
    d=pd.DataFrame(rows).sort_values('Angular separation (arcsec)').reset_index(drop=True)
    return d.iloc[0].to_dict(),d

def nearest_simbad(c,radius=2):
    s=Simbad(); s.add_votable_fields('otype','z_value','diameter'); t=s.query_region(c,radius=radius*u.arcmin); rows=[]
    if t is not None:
        for r in t:
            try:q=SkyCoord(str(r['RA']),str(r['DEC']),unit=(u.hourangle,u.deg))
            except:continue
            rows.append({'Object ID':str(r['MAIN_ID']),'Catalog':'SIMBAD','RA':q.ra.deg,'Dec':q.dec.deg,'Angular separation (arcsec)':c.separation(q).arcsec,'Redshift':sf(r['Z_VALUE']) if 'Z_VALUE' in t.colnames else np.nan,'Type':str(r['OTYPE']) if 'OTYPE' in t.colnames else '','Angular size (arcsec)':sf(r['GALDIM_MAJAXIS']) if 'GALDIM_MAJAXIS' in t.colnames else np.nan})
    if not rows:return None,pd.DataFrame()
    d=pd.DataFrame(rows).sort_values('Angular separation (arcsec)').reset_index(drop=True)
    return d.iloc[0].to_dict(),d

def details(o):
    z=sf(o.get('Redshift')); a=sf(o.get('Angular size (arcsec)')); out={'Redshift type':'Catalog value; spectroscopic/photometric flag unavailable','Stellar mass':np.nan,'Star formation rate':np.nan,'Morphology':o.get('Type','Not available')}
    if o.get('Catalog','').startswith('NASA'):
        try:
            t=Ned.query_object(o['Object ID']); r=t[0]
            a=sf(first(r,['Major Diameter','Angular Size'],a)); out['Morphology']=str(first(r,['Morphology','Type'],out['Morphology']))
        except:pass
    out['Angular size (arcsec)']=a
    if np.isfinite(z) and z>0:
        scale=Planck18.kpc_proper_per_arcmin(z).to_value(u.kpc/u.arcmin)/60*3261.563777
        out.update({'Light-travel time (Gyr)':Planck18.lookback_time(z).to_value(u.Gyr),'Comoving distance (Gly)':Planck18.comoving_distance(z).to_value(u.Glyr),'Luminosity distance (Gly)':Planck18.luminosity_distance(z).to_value(u.Glyr),'Angular diameter distance (Gly)':Planck18.angular_diameter_distance(z).to_value(u.Glyr),'Universe age at emission (Gyr)':Planck18.age(z).to_value(u.Gyr),'Scale (ly/arcsec)':scale,'Estimated physical diameter (ly)':a*scale if np.isfinite(a) else np.nan})
    return out

def fmt(v,d=4):
    x=sf(v); return 'Not available' if not np.isfinite(x) else f'{x:,.{d}f}'

def table_html(o):
    keys=['Object ID','Catalog','RA','Dec','Redshift','Redshift type','Light-travel time (Gyr)','Comoving distance (Gly)','Luminosity distance (Gly)','Angular diameter distance (Gly)','Universe age at emission (Gyr)','Angular size (arcsec)','Estimated physical diameter (ly)','Stellar mass','Star formation rate','Morphology','Type']
    rows=[]
    for k in keys:
        v=o.get(k,'Not available'); v=fmt(v) if isinstance(v,(float,np.floating)) else str(v)
        rows.append(f'<tr><th style="text-align:left;padding:5px 9px;background:#eef3f8">{html.escape(k)}</th><td style="padding:5px 9px">{html.escape(v)}</td></tr>')
    return '<table style="border-collapse:collapse;width:100%;font:13px sans-serif" border="1">'+''.join(rows)+'</table>'

def archive_links(c):
    p=f'{c.ra.deg:.7f},{c.dec.deg:.7f}'
    return {'Hubble / JWST / MAST':'https://mast.stsci.edu/portal/Mashup/Clients/Mast/Portal.html?searchQuery='+urllib.parse.quote(p),'ALMA':'https://almascience.eso.org/aq/?result_view=observation','MUSE / ESO':'https://archive.eso.org/scienceportal/home','Spitzer / IRSA':'https://irsa.ipac.caltech.edu/applications/finderchart/?locstr='+urllib.parse.quote(p),'Chandra':'https://cda.harvard.edu/chaser/','Gaia':'https://gea.esac.esa.int/archive/'}

def paper_links(o,c):
    q=o.get('Object ID') or f'{c.ra.deg:.6f} {c.dec.deg:.6f}'
    return {'NASA ADS papers':'https://ui.adsabs.harvard.edu/search/q='+urllib.parse.quote(q),'arXiv papers':'https://arxiv.org/search/?query='+urllib.parse.quote(q)+'&searchtype=all','NED references':'https://ned.ipac.caltech.edu/byname?objname='+urllib.parse.quote(q)}

header=widgets.HTML(value=f'''<div style="padding:12px 16px;border-radius:10px;background:#0b172a;color:white;font-family:sans-serif"><div style="font-size:22px;font-weight:700">INTERACTIVE MULTI-SURVEY SKY MAP — {VERSION}</div><div style="font-size:13px;opacity:.9;margin-top:4px">Original Aladin touch map · drag to pan · pinch or use +/- to zoom · real HiPS survey imagery</div></div>''')
target=widgets.Text(value=DEFAULT_TARGET,description='Target:',placeholder='RA Dec or object name',layout=widgets.Layout(width='520px'))
survey=widgets.Dropdown(options=SURVEYS,value='CDS/P/HST/GOODS/color',description='Survey:',layout=widgets.Layout(width='520px'))
fov=widgets.Dropdown(options=[('0.01° — very tight',.01),('0.03°',.03),('0.08° — HUDF/GOODS',.08),('0.25°',.25),('1°',1.),('5°',5.),('20°',20.)],value=DEFAULT_FOV,description='Field:',layout=widgets.Layout(width='300px'))
load_button=widgets.Button(description='Load Interactive Map',button_style='primary',icon='globe',layout=widgets.Layout(width='220px',height='40px'))
open_button=widgets.Button(description='Open Full Screen',icon='external-link',layout=widgets.Layout(width='180px',height='40px'))
note=widgets.HTML(value='<div style="padding:9px 12px;border-left:4px solid #ffb300;background:#fff8e1;font:13px sans-serif"><b>Coverage note:</b> DSS2, 2MASS, WISE and GALEX cover very large areas. Hubble and JWST are pointed-observation archives, so blank fields may mean no coverage.</div>')
viewer=widgets.HTML(value=viewer_html(DEFAULT_TARGET,DEFAULT_FOV,survey.value,'Hubble GOODS color'))
def reload_map(_=None):viewer.value=viewer_html(target.value,fov.value,survey.value,selected_name())
def open_full(_=None):display(widgets.HTML(value=f"<script>window.open({aladin_url(target.value,fov.value,survey.value)!r},'_blank');</script>"))
load_button.on_click(reload_map);open_button.on_click(open_full)
controls=widgets.VBox([widgets.HBox([target]),widgets.HBox([survey]),widgets.HBox([fov,load_button,open_button])])

ra_display=widgets.Text(description='Current RA:',disabled=True,layout=widgets.Layout(width='360px'))
dec_display=widgets.Text(description='Current Dec:',disabled=True,layout=widgets.Layout(width='360px'))
fov_display=widgets.Text(description='Field of View:',disabled=True,layout=widgets.Layout(width='320px'))
survey_display=widgets.Text(description='Survey:',disabled=True,layout=widgets.Layout(width='500px'))
refresh_button=widgets.Button(description='Refresh Coordinates',icon='refresh')
inspect_button=widgets.Button(description='Inspect Galaxy',button_style='success',icon='search')
save_button=widgets.Button(description='Save Galaxy',icon='save')
export_button=widgets.Button(description='Export CSV',icon='download')
notes=widgets.Textarea(description='Notes:',placeholder='Free-text notes',layout=widgets.Layout(width='100%',height='90px'))
status=widgets.HTML(); object_output=widgets.Output(); nearby_output=widgets.Output(); coverage_output=widgets.Output(); publication_output=widgets.Output(); cosmology_output=widgets.HTML()

def refresh_coordinates(_=None):
    try:
        c=parse_target(target.value);STATE['coord']=c
        ra_display.value=f"{c.ra.deg:.8f}° ({c.ra.to_string(unit=u.hour,sep=':',precision=3)})";dec_display.value=f"{c.dec.deg:.8f}° ({c.dec.to_string(unit=u.deg,sep=':',precision=3,alwayssign=True)})";fov_display.value=f'{float(fov.value):g}°';survey_display.value=selected_name();status.value='<b style="color:#1565c0">Coordinates refreshed.</b>'
    except Exception as e:status.value=f'<b style="color:#b71c1c">Coordinate error: {html.escape(str(e))}</b>'

def inspect_galaxy(_=None):
    refresh_coordinates();c=STATE.get('coord')
    if c is None:return
    inspect_button.disabled=True;status.value='<b>Querying NED/SIMBAD and calculating Planck18 cosmology…</b>'
    try:
        o,n=nearest_ned(c,max(2,float(fov.value)*30))
        if o is None:o,n=nearest_simbad(c,max(2,float(fov.value)*30))
        if o is None:raise RuntimeError('No catalog object found near the supplied center coordinates.')
        o.update(details(o));STATE['object']=o;STATE['nearby']=n.head(25)
        with object_output:clear_output();display(widgets.HTML('<h3>Nearest catalog object</h3>'+table_html(o)))
        with nearby_output:
            clear_output();display(widgets.HTML('<h3>Nearby galaxies / catalog objects</h3>'));cols=[x for x in ['Object ID','Angular separation (arcsec)','Redshift','Stellar mass'] if x in n.columns];display(n[cols].head(25).style.format(precision=5))
        with coverage_output:
            clear_output();display(widgets.HTML('<h3>Available-observation searches</h3><p>Open each live archive to confirm pointed-observation coverage.</p>'))
            for a,b in archive_links(c).items():display(widgets.HTML(f'<a target="_blank" href="{html.escape(b)}">{html.escape(a)}</a><br>'))
        with publication_output:
            clear_output();display(widgets.HTML('<h3>Publications and survey references</h3>'))
            for a,b in paper_links(o,c).items():display(widgets.HTML(f'<a target="_blank" href="{html.escape(b)}">{html.escape(a)}</a><br>'))
        cosmology_output.value=f'''<div style="padding:14px;border-radius:10px;background:#071a2b;color:white;font-family:sans-serif;line-height:1.8"><b style="font-size:18px">COSMOLOGY PANEL</b><br>You are observing this galaxy as it was: <b>{fmt(o.get('Light-travel time (Gyr)'),3)} billion years ago</b><br>Current comoving distance: <b>{fmt(o.get('Comoving distance (Gly)'),3)} billion light-years</b><br>Universe age when light was emitted: <b>{fmt(o.get('Universe age at emission (Gyr)'),3)} billion years</b><br>Scale: <b>1 arcsecond = {fmt(o.get('Scale (ly/arcsec)'),0)} light-years</b><br>Estimated galaxy diameter: <b>{fmt(o.get('Estimated physical diameter (ly)'),0)} light-years</b></div>'''
        status.value=f'<b style="color:#1b5e20">Inspection complete: {html.escape(str(o["Object ID"]))}</b>'
    except Exception as e:status.value=f'<b style="color:#b71c1c">Inspection failed: {html.escape(str(e))}</b>'
    finally:inspect_button.disabled=False

def save_galaxy(_=None):
    o=STATE.get('object');c=STATE.get('coord')
    if o is None or c is None:status.value='<b style="color:#b71c1c">Inspect a galaxy before saving.</b>';return
    row={'Date':datetime.now().astimezone().isoformat(timespec='seconds'),'RA':f'{c.ra.deg:.8f}','Dec':f'{c.dec.deg:.8f}','Object ID':o.get('Object ID','Unknown'),'Redshift':o.get('Redshift',np.nan),'Distance':o.get('Comoving distance (Gly)',np.nan),'Survey':selected_name(),'Notes':notes.value}
    pd.DataFrame([row]).to_csv(CATALOG_PATH,mode='a',header=not CATALOG_PATH.exists(),index=False);status.value=f'<b style="color:#1b5e20">Saved: {CATALOG_PATH}</b>';display(FileLink(str(CATALOG_PATH)))

def export_csv(_=None):
    if not CATALOG_PATH.exists():pd.DataFrame(columns=['Date','RA','Dec','Object ID','Redshift','Distance','Survey','Notes']).to_csv(CATALOG_PATH,index=False)
    display(FileLink(str(CATALOG_PATH)))
    try:
        from google.colab import files;files.download(str(CATALOG_PATH))
    except:pass

refresh_button.on_click(refresh_coordinates);inspect_button.on_click(inspect_galaxy);save_button.on_click(save_galaxy);export_button.on_click(export_csv);refresh_coordinates()
inspector=widgets.VBox([widgets.HTML('<h2 style="margin:12px 0 4px">GALAXY INSPECTOR</h2>'),widgets.HBox([ra_display,dec_display]),widgets.HBox([fov_display,survey_display]),widgets.HBox([refresh_button,inspect_button,save_button,export_button]),notes,status,object_output,cosmology_output,nearby_output,coverage_output,publication_output],layout=widgets.Layout(width='100%'))
display(widgets.VBox([header,controls,note,viewer,inspector],layout=widgets.Layout(width='100%')))
print(VERSION)
